<a href="https://colab.research.google.com/github/Elizabet866/lilianaI-igo-aprendizajeautomatico/blob/main/Semana_4/Notebooks/Regresi%C3%B3n_logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Actividad 2: Regresión Logística Multiclase**

**Descripción del Dataset**

El conjunto de datos contiene registros sobre el comportamiento de navegación de usuarios en una plataforma web.
- **Observaciones:** 170 sesiones de usuarios.
- **Variables predictoras ($X$):** Cuatro métricas de interacción en la plataforma: `duracion` (tiempo de sesión), `paginas` (cantidad de páginas visitadas), `acciones` (interacciones realizadas) y `valor` (puntaje de actividad).
- **Variable objetivo ($y$):** `clase` — Sistema operativo utilizado por el usuario, categorizado en tres clases discretas:
  - `0`: Windows
  - `1`: Macintosh
  - `2`: Linux

**Formulación del Problema y Pregunta de Investigación**

- **Problema:** La identificación del entorno operativo de los visitantes permite personalizar la experiencia de usuario y optimizar la compatibilidad de plataformas digitales sin depender exclusivamente de encabezados del navegador.
- **Pregunta analítica:** ¿Es posible clasificar y predecir con precisión el sistema operativo de un usuario a partir de sus métricas cuantitativas de navegación (duración, páginas, acciones y valor)?
- **Objetivo:** Entrenar y evaluar un modelo de **Regresión Logística Multiclase** (algoritmo multinomial con solver `lbfgs`) para predecir la probabilidad de pertenencia a cada una de las tres categorías de sistemas operativos.

**Importación de Librerías**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

**Carga y exploración de Datos**

In [2]:
# Carga del dataset desde repositorio público oficial
url = "https://raw.githubusercontent.com/robintux/Datasets4StackOverFlowQuestions/master/usuarios_win_mac_lin.csv"
df = pd.read_csv(url)

print("Primeras 5 filas del conjunto de datos:")
print(df.head())

print("\nInformación del DataFrame:")
print(df.info())

print("\nDistribución de las clases (0: Windows, 1: Macintosh, 2: Linux):")
print(df['clase'].value_counts())

Primeras 5 filas del conjunto de datos:
   duracion  paginas  acciones  valor  clase
0       7.0        2         4      8      2
1      21.0        2         6      6      2
2      57.0        2         4      4      2
3     101.0        3         6     12      2
4     109.0        2         6     12      2

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   duracion  170 non-null    float64
 1   paginas   170 non-null    int64  
 2   acciones  170 non-null    int64  
 3   valor     170 non-null    int64  
 4   clase     170 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 6.8 KB
None

Distribución de las clases (0: Windows, 1: Macintosh, 2: Linux):
clase
0    86
2    44
1    40
Name: count, dtype: int64


**Definición de Variables x e y**

In [3]:
# Variables predictoras (las primeras 4 columnas: duracion, paginas, acciones, valor)
X = df.iloc[:, 0:4].values

# Variable objetivo (columna 'clase')
y = df['clase'].values

print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

Dimensiones de X: (170, 4)
Dimensiones de y: (170,)


**Creación de los conjuntos de entrenamiento y prueba (80/20)**

In [4]:
# Separamos el 80% para entrenamiento y 20% para prueba manteniendo la proporción de clases (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de prueba: {len(X_test)}")

Muestras de entrenamiento: 136
Muestras de prueba: 34


**Creación y entrenamiento del modelo de Regresión Logística**

In [5]:
# Instanciamos el clasificador logístico con solver 'lbfgs' para multiclase
clasificador = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
clasificador.fit(X_train, y_train)

print("Modelo entrenado exitosamente.")

Modelo entrenado exitosamente.


**Evaluación del modelo (Exactitud, Matriz de Confusión y Reporte)**

In [6]:
y_pred = clasificador.predict(X_test)

# Cálculo de exactitud
exactitud = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy) en prueba: {exactitud:.2f}\n")

# Matriz de confusión
print("Matriz de Confusión:")
matriz = confusion_matrix(y_test, y_pred)
print(matriz)

# Reporte de clasificación detallado por clase
print("\nReporte de Clasificación detallado:")
nombres_clases = ['0 - Windows', '1 - Mac', '2 - Linux']
print(classification_report(y_test, y_pred, target_names=nombres_clases))

Exactitud (Accuracy) en prueba: 0.74

Matriz de Confusión:
[[12  0  5]
 [ 4  4  0]
 [ 0  0  9]]

Reporte de Clasificación detallado:
              precision    recall  f1-score   support

 0 - Windows       0.75      0.71      0.73        17
     1 - Mac       1.00      0.50      0.67         8
   2 - Linux       0.64      1.00      0.78         9

    accuracy                           0.74        34
   macro avg       0.80      0.74      0.73        34
weighted avg       0.78      0.74      0.73        34



**Predicción con datos nuevos hipotéticos**

In [7]:
# Simulamos dos nuevos perfiles de usuario:
# [duracion_seg, paginas_vistas, acciones, suma_valor]
nuevos_usuarios = np.array([
    [120, 3, 5, 20],     # Sesión corta
    [900, 18, 45, 210]   # Sesión larga e interactiva
])

predicciones_nuevas = clasificador.predict(nuevos_usuarios)
probabilidades = clasificador.predict_proba(nuevos_usuarios)

for i, pred in enumerate(predicciones_nuevas):
    print(f"Usuario {i+1}: Predicción = Clase {pred} ({nombres_clases[pred]}) | Probabilidades = {probabilidades[i].round(3)}")

Usuario 1: Predicción = Clase 1 (1 - Mac) | Probabilidades = [0.496 0.504 0.   ]
Usuario 2: Predicción = Clase 1 (1 - Mac) | Probabilidades = [0.159 0.841 0.   ]
